Подготовка данных для задачи обнаружения аномалий в логах

In [ ]:
import os
import json
import re
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
import random

In [ ]:
def hdfs_to_jsonl(hdfs_log_path, labels_path, output_prefix="hdfs", test_size=0.2):

    df_labels = pd.read_csv(labels_path)
    labels_dict = {}
    for _, row in df_labels.iterrows():
        labels_dict[row['BlockId']] = 1 if row['Label'] == 'Anomaly' else 0

    blk_pattern = re.compile(r'(blk_-?\d+)')
    data = []

    with open(hdfs_log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            if line[0].isdigit() and ': ' in line:
                line = line.split(': ', 1)[1]

            match = blk_pattern.search(line)
            if match:
                block_id = match.group()

                if block_id in labels_dict:
                    is_anomaly = labels_dict[block_id]

                    if is_anomaly:
                        output = "Anomaly"
                    else:
                        output = "OK"

                    data.append({
                        "input": line,
                        "output": output
                    })


    #статистика
    anomaly_count = sum(1 for d in data if d['output'] == 'Anomaly')
    print(f"OK: {len(data)-anomaly_count}, Anomaly: {anomaly_count}")

    #разделение на выборки
    labels = [1 if d['output'] == 'Anomaly' else 0 for d in data]
    train_data, test_data = train_test_split(
        data, test_size=test_size, random_state=42, stratify=labels
    )

    train_path = f"{output_prefix}_train.jsonl"
    test_path = f"{output_prefix}_test.jsonl"

    with open(train_path, 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    with open(test_path, 'w', encoding='utf-8') as f:
        for item in test_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    return train_path, test_path

In [ ]:
hdfs_log = "/content/HDFS.log"
labels = "/content/anomaly_label.csv"
hdfs_to_jsonl(hdfs_log, labels, "hdfs", test_size=0.2)